# Grids C & D — seeds 1 and 2 (Kaggle runner)

Adds the two missing seeds to the two cross-validated grids, so the numbers reported
from them become means over three seeds instead of single-seed point estimates.
Seed 0 already exists (run on Modal); this notebook does **not** re-run it.

| grid | what it is | arms | runs / seed |
|---|---|---|---|
| **C** `calibrate5` | the gamma calibration on the reference config | `A0` x {gamma=0} + {0.003,0.01,0.03,0.1,0.3} x `r_bar` | 180 |
| **D** `l1_5fold`   | the cell-encoder (L1) comparison at gamma=0 | `A0`/`A4`/`A5` | 90 |

### Two things this notebook is built to survive

**1. The weekly quota.** Measured on seed 0 (Modal, L4): grid C costs 8.2 GPU-h per
seed, of which **`cf_bc_efc` alone is 67%** — for the proxy the paper does *not*
select. A Kaggle T4 is roughly 2.5x slower than an L4, so the full two-seed plan with
both proxies is ~43 GPU-h against a 30 h/week quota: it could never finish. Dropping
`cf_bc_efc` brings it to ~16 GPU-h. `RUN_CF_BC_EFC` below turns it back on if you
want it; be aware of what it costs and what it buys.

**2. A single wedged run.** On a T4 with the preinstalled torch, one run
(`A_A0_DHFR_pcf_bc_efc_g0.003_s1_f0`) hung for almost four hours and took the whole
session with it. The same run takes 51 s on CPU at the same commit, so this is the
environment, not the code. Every chunk therefore runs in a **subprocess with a hard
timeout**: a run that wedges costs one chunk, is quarantined, and the session
continues. The smoke test below catches the same pathology in a minute instead of
four hours.

Resumption is by `run_id`: `ResultsStore.has_run` skips anything already stored, so
re-running this notebook never repeats work. To carry a session forward, attach its
output as an input dataset and set `RESUME_FROM`.

**Before running:** Settings -> Accelerator = GPU, Internet = **On**.


In [ ]:
# =====================================================================
# CONFIG - every knob of this notebook is in this cell.
# =====================================================================
REPO_URL = "https://github.com/AlGoRythm3000/Differentiable-Motif-Discovery.git"
BRANCH   = "main"
REPO_DIR = "/kaggle/working/repo"
OUT_DIR  = "/kaggle/working/results"
DATA_DIR = "/kaggle/working/datasets"

# Seed 0 is already done (Modal). These two turn each grid-C / grid-D number
# into a mean +- std over three seeds.
SEEDS = [1, 2]
RUN_GRID_C = True     # calibrate5: A0 x gamma, 5-fold
RUN_GRID_D = True     # l1_5fold:   A0/A4/A5 at gamma=0, 5-fold

CALIBRATE_GAMMAS = [0.0, 0.003, 0.01, 0.03, 0.1, 0.3]

# `cf_bc_efc` is 67% of grid C's cost, for the proxy the paper does not select.
# Turning it on roughly triples the plan and it will NOT fit the weekly quota in
# one week. The scientific cost of leaving it off is real and worth naming: the
# three-seed result then strengthens the choice of GAMMA only, and the r_bar vs
# cf_bc_efc proxy comparison stays single-seed.
RUN_CF_BC_EFC = False
CALIBRATE_PROXIES = ["r_bar", "cf_bc_efc"] if RUN_CF_BC_EFC else ["r_bar"]

DATASETS = ["synthetic_bottleneck", "MUTAG", "PROTEINS", "IMDB-BINARY", "ENZYMES", "DHFR"]
CV_FOLDS = 5

# --- session limits --------------------------------------------------
WALL_BUDGET_H = 11.0   # Kaggle kills a GPU session at 12 h; stop before that.
CHUNK = 10             # runs per subprocess
# Hard ceiling per chunk. The slowest legitimate chunk on seed 0 was ten
# PROTEINS/cf_bc_efc runs at ~234 s each, so ~40 min; 75 min leaves room for a
# slower GPU while still catching a wedge in a bounded time.
CHUNK_TIMEOUT_S = 75 * 60

SHARD_ID   = 0
NUM_SHARDS = 1

# Attach a previous session's output as an input dataset, then point this at the
# directory holding its runs.csv. Leave None for the first session.
RESUME_FROM = None     # e.g. "/kaggle/input/dmd-seeds-cd-session1"

INSTALL_DEPS = True


In [ ]:
# =====================================================================
# SETUP - dependencies, clone at the pinned branch
# =====================================================================
import os, subprocess, sys

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_geometric", "networkx", "pyyaml"], check=False)
    # topomodelx/toponetx pin numpy<2 and pull `pyg-nightly`, a SECOND
    # distribution of the `torch_geometric` package. Letting their resolver run
    # mixes files from both distributions in site-packages and produces
    # "partially initialized module 'torch_geometric' has no attribute 'typing'"
    # at import time. --no-deps installs them without touching PyG or numpy.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "topomodelx", "toponetx"], check=False)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())


In [ ]:
# =====================================================================
# ENVIRONMENT LOG
# =====================================================================
import torch

from tools.experiment_grid import GridConfig, build_plan, environment_info
from tools.results_store import ResultsStore

ENV = environment_info(REPO_DIR)
ENV.update({"branch": BRANCH, "shard": f"{SHARD_ID}/{NUM_SHARDS}", "runner": "kaggle",
            "seeds": ",".join(str(s) for s in SEEDS),
            "proxies": ",".join(CALIBRATE_PROXIES)})
for key, value in ENV.items():
    print(f"{key:>18}: {value}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{'device':>18}: {DEVICE}")
if DEVICE == "cuda":
    print(f"{'gpu':>18}: {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability(0)
    print(f"{'compute cap':>18}: sm_{cap[0]}{cap[1]}")
    # A wheel with no kernels for this architecture makes CUDA JIT from PTX at
    # every first launch, which is one of the ways a run turns from 51 s into
    # hours. Worth knowing before the smoke test, not after.
    print(f"{'built for':>18}: {', '.join(torch.cuda.get_arch_list())}")
else:
    print("WARNING: no GPU visible - set Settings -> Accelerator to GPU.")


In [ ]:
# =====================================================================
# SMOKE TEST - time the OSq path on GPU against CPU, on the arm that wedged.
#
# The gamma=0 runs never touch the conjugate-gradient kernels; the first
# gamma>0 run does, which is exactly where the four-hour stall happened. This
# cell pays ~2 minutes to find that out instead of most of a session.
# =====================================================================
import time

from tools.experiment_grid import RunSpec, load_arm, run_single
from tools.config_loader import load_all_configs

ALL_CONFIGS = load_all_configs(f"{REPO_DIR}/configs")
SMOKE = {}
for dev in ([DEVICE] if DEVICE == "cpu" else ["cuda", "cpu"]):
    cfg = GridConfig(datasets=["MUTAG"], configs_dir=f"{REPO_DIR}/configs",
                     seeds=[1], cv_folds=CV_FOLDS, grad_clip=1.0,
                     data_root=DATA_DIR, device=dev, time_budget_s=None)
    cfg.commit_sha = ENV.get("commit_sha", "unknown")
    ds, nfeat, nclass, _ = load_arm("MUTAG", cfg)
    spec = RunSpec("A", "A0", "MUTAG", 0.03, 1, fold=0, proxy=CALIBRATE_PROXIES[0])
    t0 = time.time()
    run_single(spec, cfg, ALL_CONFIGS["A0"], ds, nfeat, nclass)
    SMOKE[dev] = time.time() - t0
    print(f"  {dev:>6}: {SMOKE[dev]:6.1f}s")

if "cuda" in SMOKE and "cpu" in SMOKE and SMOKE["cuda"] > 3 * SMOKE["cpu"]:
    # Not a warning to scroll past: at this ratio the GPU plan cannot finish,
    # and the CPU is the faster machine for this workload.
    print(f"\nGPU is {SMOKE['cuda'] / SMOKE['cpu']:.1f}x SLOWER than CPU on the "
          f"conjugate-gradient path.\nFalling back to CPU for the whole grid.")
    DEVICE = "cpu"
print(f"\nrunning the grid on: {DEVICE}")


In [ ]:
# =====================================================================
# PLAN - build grids C and D exactly as seed 0 was built, then filter
# =====================================================================
import hashlib
from collections import Counter

COMMON = dict(datasets=DATASETS, configs_dir=f"{REPO_DIR}/configs",
              seeds=SEEDS, cv_folds=CV_FOLDS, grad_clip=1.0,
              data_root=DATA_DIR, device=DEVICE,
              # Left None on purpose: the projection-based trimmer drops whole
              # tiers on one global mean seconds-per-run and has silently
              # deleted two thirds of a grid before. The wall-clock check in the
              # run cell is the cutoff instead, and it drops nothing - it stops,
              # and the next session resumes.
              time_budget_s=None)

CONFIG_C = GridConfig(gammas=CALIBRATE_GAMMAS, proxies=CALIBRATE_PROXIES, **COMMON)
plan_c = [s for s in build_plan(CONFIG_C) if s.config_id == "A0"] if RUN_GRID_C else []

CONFIG_D = GridConfig(gammas=[0.0], **COMMON)
plan_d = [s for s in build_plan(CONFIG_D)
          if s.config_id in ("A0", "A4", "A5")] if RUN_GRID_D else []

# A0 at gamma=0 is the same run_id in both grids: stored once, read by both.
seen, plan = set(), []
for spec in plan_c + plan_d:
    if spec.run_id not in seen:
        seen.add(spec.run_id)
        plan.append(spec)

CONFIG = CONFIG_C if RUN_GRID_C else CONFIG_D
CONFIG.commit_sha = ENV.get("commit_sha", "unknown")
print(f"grid C : {len(plan_c):>4} specs\ngrid D : {len(plan_d):>4} specs")
print(f"union  : {len(plan):>4} distinct run_ids")


In [ ]:
# =====================================================================
# RESUME + QUARANTINE + SHARD + ORDER
# =====================================================================
import json, shutil
from pathlib import Path

QUARANTINE = Path(OUT_DIR) / "quarantine.json"

if RESUME_FROM:
    src = Path(RESUME_FROM) / "runs.csv"
    if not src.exists():
        raise SystemExit(f"RESUME_FROM={RESUME_FROM} has no runs.csv")
    dst = Path(OUT_DIR) / "runs.csv"
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f"resumed from {src}")
    q = Path(RESUME_FROM) / "quarantine.json"
    if q.exists() and not QUARANTINE.exists():
        shutil.copy(q, QUARANTINE)

store = ResultsStore(OUT_DIR)
done = store.existing_run_ids()
plan = [s for s in plan if s.run_id not in done]
print(f"already stored : {len(done)}")

# A run that wedged a previous chunk is retried once, then left alone: retrying
# it forever is how one pathological spec eats every session in turn.
quarantined = json.loads(QUARANTINE.read_text()) if QUARANTINE.exists() else {}
blocked = {k for k, n in quarantined.items() if n >= 2}
if blocked:
    plan = [s for s in plan if s.run_id not in blocked]
    print(f"quarantined    : {len(blocked)} (timed out twice)")

if NUM_SHARDS > 1:
    plan = [s for s in plan
            if int(hashlib.sha1(s.run_id.encode()).hexdigest(), 16) % NUM_SHARDS == SHARD_ID]
    print(f"shard {SHARD_ID}/{NUM_SHARDS}  : {len(plan)}")

# Cheapest dataset first (mean seconds per run, measured on seed 0), so a
# session that runs out of time leaves the most WHOLE runs behind. Chunks are
# grouped by dataset, which also means each subprocess loads one dataset.
COST_ORDER = ["MUTAG", "DHFR", "synthetic_bottleneck", "IMDB-BINARY", "ENZYMES", "PROTEINS"]
plan.sort(key=lambda s: (COST_ORDER.index(s.dataset) if s.dataset in COST_ORDER else 99,
                         s.run_id))
print(f"to run         : {len(plan)}")
for ds, n in sorted(Counter(s.dataset for s in plan).items(),
                    key=lambda kv: COST_ORDER.index(kv[0])):
    print(f"   {ds:<22} {n}")


In [ ]:
# =====================================================================
# WORKER - one chunk, in its own process.
#
# A subprocess rather than a thread because a wedged CUDA call cannot be
# interrupted from Python: only killing the process gets the session back.
# Everything crosses the boundary as plain JSON, and the child appends to the
# same append-only store, so a chunk killed halfway keeps the runs it finished.
# =====================================================================
WORKER = """
import json, sys
sys.path.insert(0, sys.argv[1])
from tools.experiment_grid import GridConfig, RunSpec, run_grid
from tools.results_store import ResultsStore

payload = json.load(open(sys.argv[2]))
config = GridConfig(**payload["config"])
config.commit_sha = payload["commit_sha"]
plan = [RunSpec(**s) for s in payload["specs"]]
run_grid(config, ResultsStore(payload["out_dir"]), plan=plan, verbose=True)
"""
Path(f"{OUT_DIR}/_worker.py").parent.mkdir(parents=True, exist_ok=True)
Path(f"{OUT_DIR}/_worker.py").write_text(WORKER)

CONFIG_KWARGS = dict(datasets=DATASETS, configs_dir=f"{REPO_DIR}/configs",
                     seeds=SEEDS, cv_folds=CV_FOLDS, grad_clip=1.0,
                     data_root=DATA_DIR, device=DEVICE, time_budget_s=None,
                     gammas=CALIBRATE_GAMMAS, proxies=CALIBRATE_PROXIES)
print(f"wrote {OUT_DIR}/_worker.py")


In [ ]:
# =====================================================================
# RUN
# =====================================================================
import time

# Recorded AFTER the smoke test, so a row is attributable to the device that
# actually ran it rather than the one the session started with.
ENV["device"] = DEVICE
ENV["smoke_seconds"] = json.dumps({k: round(v, 1) for k, v in SMOKE.items()})
store.write_env(ENV)
started = time.time()
budget_s = WALL_BUDGET_H * 3600
timed_out, stopped_early = [], False

for offset in range(0, len(plan), CHUNK):
    elapsed = time.time() - started
    if elapsed > budget_s:
        stopped_early = True
        print(f"\n=== wall budget reached after {elapsed/3600:.2f} h, "
              f"{len(plan) - offset} specs left - stopping cleanly ===")
        break
    chunk = plan[offset:offset + CHUNK]
    print(f"\n--- {offset + 1}-{offset + len(chunk)} / {len(plan)} "
          f"[{chunk[0].dataset}] ({elapsed/3600:.2f} h elapsed) ---", flush=True)

    payload = {"config": CONFIG_KWARGS, "commit_sha": ENV.get("commit_sha", "unknown"),
               "out_dir": OUT_DIR,
               "specs": [{"tier": s.tier, "config_id": s.config_id, "dataset": s.dataset,
                          "gamma": s.gamma, "seed": s.seed, "fold": s.fold,
                          "proxy": s.proxy, "sparsity_weight": s.sparsity_weight}
                         for s in chunk]}
    Path(f"{OUT_DIR}/_chunk.json").write_text(json.dumps(payload))
    try:
        subprocess.run([sys.executable, f"{OUT_DIR}/_worker.py", REPO_DIR,
                        f"{OUT_DIR}/_chunk.json"], timeout=CHUNK_TIMEOUT_S, check=False)
    except subprocess.TimeoutExpired:
        # The chunk is gone, but the store kept whatever it finished; only the
        # specs still missing are charged with the timeout.
        store = ResultsStore(OUT_DIR)
        stuck = [s.run_id for s in chunk if s.run_id not in store.existing_run_ids()]
        for rid in stuck:
            quarantined[rid] = quarantined.get(rid, 0) + 1
        QUARANTINE.write_text(json.dumps(quarantined, indent=1))
        timed_out += stuck
        print(f"  !! chunk exceeded {CHUNK_TIMEOUT_S}s - killed. "
              f"{len(stuck)} run(s) quarantined: {stuck[:3]}", flush=True)
    store = ResultsStore(OUT_DIR)

print(f"\n{'=' * 60}")
print(f"elapsed      : {(time.time() - started)/3600:.2f} h")
print(f"stored       : {len(store.existing_run_ids())}")
print(f"timed out    : {len(timed_out)}")
print(f"stopped early: {stopped_early}")


In [ ]:
# =====================================================================
# SUMMARY + ZIP
# =====================================================================
from collections import defaultdict

rows = store.read_runs()
ok = [r for r in rows if r.get("status") == "ok"]
print(f"{len(ok)}/{len(rows)} rows ok in {OUT_DIR}/runs.csv\n")

bad = [r for r in rows if r.get("status") != "ok"]
if bad:
    print("FAILED RUNS:")
    for r in bad[:20]:
        print(f"   {r['run_id']:<52} {str(r.get('error'))[:70]}")
    print()
if timed_out:
    print("TIMED OUT (quarantined, retried once before being left alone):")
    for rid in timed_out[:20]:
        print(f"   {rid}")
    print()

by = defaultdict(list)
for r in ok:
    by[(r["seed"], r["config_id"], r.get("osq_proxy") or "none")].append(r)
print(f"{'seed':>6}{'config':>9}{'proxy':>12}{'n':>6}{'test_acc':>12}")
for (seed, cid, px), rs in sorted(by.items()):
    accs = [float(r["test_acc"]) for r in rs if r.get("test_acc") not in (None, "")]
    mean = sum(accs) / len(accs) if accs else float("nan")
    print(f"{seed:>6}{cid:>9}{px:>12}{len(rs):>6}{mean:>12.4f}")

archive = store.zip("/kaggle/working/dmd_seeds_cd.zip")
print(f"\nwrote {archive}")
print("""
NEXT SESSION
------------
1. "Save Version" -> this notebook's output becomes a dataset.
2. New session: "+ Add Data" -> Your Datasets -> that output.
3. Set RESUME_FROM to its path. Everything stored is skipped.

BACK IN THE REPO
----------------
Unzip into results/seeds_cd/, then merge with seed 0 and re-run the analysis.
""")
